---
## Sección 8. Refactorización: de código espagueti a clases

Una agencia de viajes tiene un programa que **funciona**, pero está escrito todo seguido. Tiene listas con posiciones difíciles de recordar, muchos `if/elif` según el tipo de reserva y cálculos repetidos. Cada vez que la agencia quiere vender un nuevo tipo de reserva hay que tocar el programa en varios sitios.

Tu tarea es **reorganizarlo usando clases, objetos, herencia y polimorfismo**, sin cambiar lo que muestra por pantalla.

### Parte A. Analizar el código original

Copia el código, ejecútalo y anota su salida. **No lo modifiques todavía.**

```python
# Programa original de la agencia de viajes
# Cada reserva: [tipo, cliente, cantidad, precio_unitario, destino]
reservas = [
    ["vuelo", "Ana López", 2, 120.0, "Madrid-Roma"],
    ["hotel", "Luis Pérez", 3, 85.0, "Hotel Mar Azul"],
    ["tren", "Marta Gil", 1, 45.0, "Madrid-Sevilla"],
    ["vuelo", "Pablo Ruiz", 1, 300.0, "Madrid-Nueva York"],
    ["hotel", "Eva Sanz", 2, 140.0, "Hotel Centro"],
]

total = 0
for r in reservas:
    if r[0] == "vuelo":
        precio = r[2] * r[3]
        precio = precio + 25
        if r[3] > 200:
            precio = precio + r[2] * 30
        print("VUELO | " + r[1] + " | " + r[4] + " | " + str(r[2]) + " pasajeros | " + str(round(precio, 2)) + " €")
    elif r[0] == "hotel":
        precio = r[2] * r[3]
        if r[2] >= 3:
            precio = precio * 0.9
        precio = precio + r[2] * 1.5
        print("HOTEL | " + r[1] + " | " + r[4] + " | " + str(r[2]) + " noches | " + str(round(precio, 2)) + " €")
    elif r[0] == "tren":
        precio = r[2] * r[3]
        print("TREN | " + r[1] + " | " + r[4] + " | " + str(r[2]) + " billetes | " + str(round(precio, 2)) + " €")
    total = total + precio

print("TOTAL AGENCIA: " + str(round(total, 2)) + " €")

# Contar reservas de más de 200 €: hay que repetir todos los cálculos
caras = 0
for r in reservas:
    if r[0] == "vuelo":
        precio = r[2] * r[3] + 25
        if r[3] > 200:
            precio = precio + r[2] * 30
    elif r[0] == "hotel":
        precio = r[2] * r[3]
        if r[2] >= 3:
            precio = precio * 0.9
        precio = precio + r[2] * 1.5
    elif r[0] == "tren":
        precio = r[2] * r[3]
    if precio > 200:
        caras = caras + 1

print("Reservas de más de 200 €: " + str(caras))
```

Las reglas de precio que esconde el código son:

| Tipo | Cantidad significa | Cálculo del precio |
| --- | --- | --- |
| Vuelo | pasajeros | pasajeros × precio + 25 € de tasas. Si el billete cuesta más de 200 €, se suman 30 € por pasajero. |
| Hotel | noches | noches × precio. Con 3 noches o más, un 10 % de descuento. Después se suma 1.5 € por noche de tasa turística. |
| Tren | billetes | billetes × precio. |

Antes de programar, responde:

45. ¿Qué significan `r[2]` y `r[3]`? ¿Es fácil saberlo sin leer el comentario?
46. ¿En cuántos sitios distintos está escrita la regla del precio de un hotel?
47. Si mañana la agencia vende **cruceros**, ¿cuántas partes del programa habría que tocar?

✍️ **Respuestas:**



### Parte B. Crear la superclase

Crea una clase `Reserva` con:

- **Atributos:** `cliente`, `destino`, `cantidad` y `precio_unitario`.
- **Método `precio_base()`:** devuelve cantidad × precio unitario.
- **Método `precio_total()`:** en la superclase devuelve el precio base.
- **Método `unidad()`:** devuelve el texto de la unidad, por ejemplo `"unidades"`.
- **Método `__str__`:** devuelve la línea completa con el formato del programa original. El tipo (`VUELO`, `HOTEL`…) se obtiene a partir del nombre de la clase, y la unidad, con `unidad()`.

### Parte C. Crear las subclases

Crea `ReservaVuelo`, `ReservaHotel` y `ReservaTren`. Cada una debe:

- sobrescribir `unidad()`;
- sobrescribir `precio_total()` **solo si lo necesita**, reutilizando `precio_base()` o `super().precio_total()` en lugar de repetir la multiplicación.
Pista para el tipo: si la clase se llama `ReservaVuelo`, puedes obtener `"VUELO"` con `type(self).__name__.replace("Reserva", "").upper()`.

### Parte D. Crear una clase que agrupe las reservas (composición)

Crea una clase `Agencia` que **contenga** una lista de reservas y tenga estos métodos:

- `agregar(reserva)`;
- `mostrar()`, que imprime cada reserva;
- `total()`, que devuelve la suma de los precios totales;
- `contar_superiores_a(importe)`, que devuelve cuántas reservas superan ese importe.

### Parte E. Reescribir el programa principal

```python
agencia = Agencia()
agencia.agregar(ReservaVuelo("Ana López", "Madrid-Roma", 2, 120.0))
agencia.agregar(ReservaHotel("Luis Pérez", "Hotel Mar Azul", 3, 85.0))
agencia.agregar(ReservaTren("Marta Gil", "Madrid-Sevilla", 1, 45.0))
agencia.agregar(ReservaVuelo("Pablo Ruiz", "Madrid-Nueva York", 1, 300.0))
agencia.agregar(ReservaHotel("Eva Sanz", "Hotel Centro", 2, 140.0))

agencia.mostrar()
print(f"TOTAL AGENCIA:{round(agencia.total(), 2)} €")
print(f"Reservas de más de 200 €:{agencia.contar_superiores_a(200)}")
```

**Salida esperada** (idéntica a la del programa original)

```text
VUELO | Ana López | Madrid-Roma | 2 pasajeros | 265.0 €
HOTEL | Luis Pérez | Hotel Mar Azul | 3 noches | 234.0 €
TREN | Marta Gil | Madrid-Sevilla | 1 billetes | 45.0 €
VUELO | Pablo Ruiz | Madrid-Nueva York | 1 pasajeros | 355.0 €
HOTEL | Eva Sanz | Hotel Centro | 2 noches | 283.0 €
TOTAL AGENCIA: 1182.0 €
Reservas de más de 200 €: 4
```

### Parte F. Ampliación

La agencia empieza a vender **cruceros**. La cantidad son los camarotes y el precio es camarotes × precio más un 12 % de tasas portuarias. Crea `ReservaCrucero` y añade esta reserva:

```python
agencia.agregar(ReservaCrucero("Jorge Vidal", "Crucero Mediterráneo", 1, 900.0))
```

**Salida esperada** (solo las líneas nuevas o que cambian)

```text
CRUCERO | Jorge Vidal | Crucero Mediterráneo | 1 camarotes | 1008.0 €
TOTAL AGENCIA: 2190.0 €
Reservas de más de 200 €: 5
```

### Restricciones

- En el programa final **no puede quedar ningún `if/elif` que pregunte por el tipo de reserva**.
- No se puede usar `isinstance()` para decidir cómo calcular el precio.
- El cálculo de cada tipo de reserva debe estar escrito **en un único sitio**.
- Para añadir el crucero **no debes modificar** ni `Reserva` ni `Agencia`.
**Pregunta final:** compara tu respuesta a la pregunta 3 de la Parte A con lo que has tenido que hacer en la Parte F. ¿Qué ha cambiado?

✍️ **Respuestas:**

